# Converting exported and scraped data into a harmonised format for comparision
This notebook provides the code to convert exported `.csv` files from the [HMA-EMA Catalogue of real-world data studies](https://catalogues.ema.europa.eu/search?f%5B0%5D=content_type%3Adarwin_study) and compare them to the scraped `.csv` files (from the same database).

<small>**NOTE:** We did use this notebook to figure out differences between our own extraction method and EMAs methods.</small>

<small>These are the biggest differences:
1. EMAs `.csv` contains **document names** vs. the **document URL** in our version
    + Interestingly it is not possible to construct the URL from the document name even though they used a simple urlquoted version of the name in their URLs at first
        + This is demonstrated in the `.ipynb` used for document download
    + That's because there were some URLs with extra spaces and other small changes so that is wasn't possible to do this
    + They are also using different base URLs for the document URLs now
1. EMAs `.csv` had a bug which cuts off text between < and > (This is likely a HTML/XML parsing error)
1. EMAs `.csv` used `, ` as a delimiter, which causes problems for text which already contains this delimiter
</small>

First we import `pandas`:

In [4]:
import pandas as pd

## Reading, cleaning and tranforming exported data

In [ ]:
exported = pd.read_csv('./data/exported_ema-rwd_2024-02-21T23-20-00+00-00.csv')
exported

We can list all colum names by uncommmenting the last line of code. Note that `data_source_types` and `data_source_types_other` are not exported by the tool as of $\texttt{21-02-2024}$.

In [ ]:
# NOTE: Not included
# data_source_types
# data_source_types_other
print('\n'.join(exported.columns.tolist()))

There were no Protocol, Result or Other documents Urls at the time of extraction.

In [ ]:
exported[['Protocol.1', 'Study report.1', 'Study, other information.1']].value_counts()

There were no Summary Results at the time of extraction.

In [ ]:
exported['Summary results'].value_counts()

There were no alternative study ids either.

In [ ]:
exported[['Study ID, other', 'Study ID, other.1']].value_counts()

Now we can copy the relevant parts of this list to create a rename mapping for the columns

In [8]:
rename_cols = {
 'Title': 'title',
 'URL LINK': 'url',
 'First published': 'registration_date',
 'Updated': 'update_date',
 'PURI': 'puri',
 'EU PAS number': 'eu_pas_register_number',
 'Study countries': 'countries',
 'Study description': 'description',
 'Study status': 'state',
 'Institution conducting the study': 'lead_institution_encepp',
 'Institution conducting the study if not in the list': 'lead_institution_not_encepp',
 'Additional institutions': 'additional_institutions_encepp',
 'Additional institutions if not in the list': 'additional_institutions_not_encepp',
 'Networks conducting the study': 'networks_encepp',
 'Additional networks if not in the list': 'networks_not_encepp',
 'Date when funding contract was signed (Planned)': 'funding_contract_date_planed',
 'Date when funding contract was signed (Actual)': 'funding_contract_date_actual',
 'Study start date (Planned)': 'data_collection_date_planed',
 'Study start date (Actual)': 'data_collection_date_actual',
 'Data analysis start date (Planned)': 'data_analysis_date_planed',
 'Data analysis start date (Actual)': 'data_analysis_date_actual',
 'Date of interim report, if expected (Planned)': 'iterim_report_date_planed',
 'Date of interim report, if expected (Actual)': 'iterim_report_date_actual',
 'Date of final study report (Planned)': 'final_report_date_planed',
 'Date of final study report (Actual)': 'final_report_date_actual',
 'Source of funding': 'funding_sources',
 'More details on source of funding': 'funding_details',
 'Protocol': 'protocol_document_name',
 'Was the study required by a regulatory body?': 'requested_by_regulator',
 'Is the study required by a Risk Management Plan (RMP)?': 'risk_management_plan',
 'Regulatory procedure number': 'regulatory_procedure_number',
 'Study topic': 'study_topic',
 'Study topic, other': 'study_topic_other',
 'Study type': 'study_type',
 'If ‘Not applicable’, further details on the study type': 'study_type_other',
 'Scope of the study': 'non_interventional_scopes',
 'If ‘other’, further details on the scope of the study': 'non_interventional_scopes_other',
 'Non-interventional study design': 'non_interventional_study_design',
 'Non-interventional study design, other': 'non_interventional_study_design_other',
 'Name of medicine': 'substance_brand_name',
 'Name of medicine, other': 'substance_brand_name_other',
 'Study drug International non-proprietary name (INN) or common name': 'substance_inn',
 'Anatomical Therapeutic Chemical (ATC) code': 'substance_atc',
 'Medicinal condition to be studied': 'medical_conditions',
 'Additional medical condition(s)': 'additional_medical_conditions',
 'Population age groups': 'age_population',
 'Special population of interest': 'special_population',
 'Special population of interest, other' : 'special_population_other',
 'Estimated number of subjects': 'number_of_subjects',
 'Outcomes': 'outcomes',
 'Results tables': 'result_tables_name',
 'Study report': 'result_document_name',
 'Study, other information': 'other_documents_name',
 'Study publications': 'references',
 'Data source(s) ': 'data_sources_registered_with_encepp',
 'Other linked data sources ': 'data_sources_not_registered_with_encepp',
 'Check conformance': 'check_conformance',
 'Check completeness': 'check_completeness',
 'Check stability': 'check_stability',
 'Check logical consistency': 'check_logical_consistency',
 'Data characterisation conducted': 'conducted_data_characterisation'
}

We can now rename the columns and drop the columns missing in the rename map:

In [ ]:
harmonised_exported = exported.filter(items=rename_cols.keys()).rename(columns=rename_cols)
harmonised_exported

At this point we can use the same naming conventions for the scraped and exported data

In [ ]:
cleaned = harmonised_exported.copy()

We will now transform the columns.

* `eu_pas_register_number` will be the new numeric index

* All dates should be converted into date objects

* We will split the array string with the delimiter `, `, sort the array and join this sorted array with the delimiter `; ` used in the scraped data

In [10]:
cleaned['eu_pas_register_number'] = cleaned['eu_pas_register_number'].str[5:].astype(int)
cleaned = cleaned.set_index('eu_pas_register_number').sort_index()

In [11]:
date_cols = cleaned.columns[cleaned.columns.str.contains('date')]
for col in date_cols:
    cleaned[col] = pd.to_datetime(cleaned[col], format='%d/%m/%Y', exact=False)

cleaned.filter(like='date')

In [12]:
# We will handle 'countries' seperatly to fix the problem with the ', ' delimiter appearing in country names
array_fields = ['additional_institutions_encepp', 'age_population', 'data_sources_registered_with_encepp', 
                'funding_sources', 'medical_conditions', 'networks_encepp', 'non_interventional_scopes', 
                'non_interventional_study_design', 'other_documents_name', 'references', 'special_population',
                'study_topic', 'substance_atc', 'substance_brand_name', 'substance_inn']
# Venezuela, Bolivarian Republic of; 
for field in array_fields:
    #cleaned[field] = cleaned[field].str.replace(', ', '; ')
    cleaned[field] = cleaned[field].str.split(', ').apply(lambda x : sorted(x) if type(x) is list else x).str.join('; ')

cleaned.filter(items=array_fields)

## Finding unique dummie values
We can use the code below to find all possible values for a field. These should be the same as in this [document](https://catalogues.ema.europa.eu/system/files/2024-01/Study_Questionnaire_Offline.pdf), but this is not the case as of 21-02-2024.

In [13]:
cleaned['funding_sources'].str.get_dummies('; ').columns.tolist()

In [14]:
cleaned['study_topic'].str.get_dummies('; ').columns.tolist()

In [15]:
cleaned['non_interventional_scopes'].str.get_dummies('; ').columns.tolist()

In [16]:
cleaned['non_interventional_study_design'].str.get_dummies('; ').columns.tolist()

In [17]:
cleaned['age_population'].str.get_dummies('; ').columns.tolist()

In [18]:
cleaned['special_population'].str.get_dummies('; ').columns.tolist()

Here is the final Dataframe:

In [20]:
cleaned.sort_index(axis='columns')

Now we will load the scraped data

In [21]:
scraped = pd.read_csv('./data/scraped_ema-rwd_2024-02-21T22-22-05+00-00.csv').set_index('eu_pas_register_number').sort_index()
scraped

We can now fix the `countries` field. First we get the list of all country values containing a comma.

In [22]:
comma_countries = [c for c in scraped['countries'].str.get_dummies('; ').columns.values if ',' in c]
comma_countries

Next we will replace the comma with another character split, sort and join or string and then restore the comma.

In [23]:
for c in comma_countries:
    cleaned['countries'] = cleaned['countries'].str.replace(c, c.replace(',', '|'), regex=False)

cleaned['countries'] = cleaned['countries'].str.split(', ').apply(lambda x : sorted(x) if type(x) is list else x).str.join('; ')

for c in comma_countries:
    cleaned['countries'] = cleaned['countries'].str.replace(c.replace(',', '|'), c, regex=False)

cleaned['countries'].str.get_dummies('; ').columns.tolist()[:10]

## Comparing the data
We will now compare the data. The Dataframes need to have the same index and columns in order to use `pd.compare`. We will use `intersection` to find the shared index and column values.

In [24]:
shared_cols = cleaned.columns.intersection(scraped.columns)
print('Amount of shared cols: ', len(shared_cols))
print('Amount of cols in cleaned: ', len(cleaned.columns))
print('Amount of cols in scraped: ', len(scraped.columns))
print('Symmetric differences: ', scraped.columns.symmetric_difference(cleaned.columns))

shared_indices = scraped.index.intersection(cleaned.index)
print('Amount of shared indices: ', len(shared_indices))
print('Amount of indices in cleaned: ', len(cleaned.index))
print('Amount of indices in scraped: ', len(scraped.index))
print('Symmetric differences: ', scraped.index.symmetric_difference(cleaned.index))

Okay. Now we can compare the Dataframes. We will exclude substance_atc and url, as we know that these are different, because they extract different informations.

In [25]:
# We know that there are differences in substance_atc and url
interesting_cols = shared_cols.difference(['substance_atc', 'url'])
df_c = cleaned.filter(items=interesting_cols).filter(items=shared_indices, axis='index').sort_index().sort_index(axis='columns')
df_s = scraped.filter(items=interesting_cols).filter(items=shared_indices, axis='index').sort_index().sort_index(axis='columns')

comparison = df_c.compare(df_s, result_names=('exported', 'scraped'))
comparison

## Exporting
There seems to be a problem, when we try to export the exported `.csv` as a `.xlsx`.

It is related to these illegal characters:

In [26]:
from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE
ILLEGAL_CHARACTERS_RE

We can find the entries containing these characters...

In [27]:
matches = []
for col in cleaned:
    m = cleaned.loc[cleaned[col].astype(str).str.contains(ILLEGAL_CHARACTERS_RE), col]
    if not m.empty:
        matches.append(m)
matches

...and display the value

In [28]:
cleaned.loc[6435].title

We can still export the data if we encode and decode these ASCII-control characters like so:

In [29]:
cleaned.reset_index().map(lambda x: x.encode('unicode_escape').decode('utf-8') if isinstance(x, str) and ILLEGAL_CHARACTERS_RE.search(x) else x).to_excel('converted_ema-rwd_2024-02-21.xlsx', sheet_name='PAS')
comparison.map(lambda x: x.encode('unicode_escape').decode('utf-8') if isinstance(x, str) and ILLEGAL_CHARACTERS_RE.search(x) else x).to_excel('dataset_comparison_2024-02-21.xlsx', sheet_name='compare')